# AncestryClassifier — Colab Training

Run preprocessing locally with Snakemake first:
```bash
snakemake --cores 4 prepare_training_data simulate_admixed
```
Upload `data/dataset.h5` and `data/admixed_test.h5` to Google Drive, then set `DRIVE_DIR` below.

**After any runtime restart: re-run Cell 1 (config) before running any other cell.**

In [1]:
# ── Cell 1: config — re-run this first after every runtime restart ─────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR     = '/content/drive/MyDrive/gene461'          # <-- change if needed
REPO          = '/content/gene_461_final_project'
WINDOW_SIZE   = 1000

DATA_H5       = f'{DRIVE_DIR}/dataset.h5'
ADMIXED_H5    = f'{DRIVE_DIR}/admixed_test.h5'
CKPT_OUT      = f'{DRIVE_DIR}/checkpoints/best_model.pt'
CONFUSION_OUT = f'{DRIVE_DIR}/confusion_matrix.png'
KARYOGRAM_OUT = f'{DRIVE_DIR}/lai_karyogram.png'

Mounted at /content/drive


In [2]:
# ── Cell 2: one-time setup (clone repo + install deps) ────────────────────
import subprocess, os

if not os.path.exists(REPO):
    subprocess.run(
        ['git', 'clone', 'https://github.com/aszatrowski/gene_461_final_project', REPO],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

%pip install -q torch h5py numpy pandas scikit-learn matplotlib wandb

In [3]:
# ── Cell 3: verify GPU ─────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA available: True
GPU: Tesla T4


In [ ]:
# ── Cell 4: baseline train (needs Cell 1) ─────────────────────────────────
import os
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

!python {REPO}/scripts/train.py \
    --data        {DATA_H5}    \
    --output      {CKPT_OUT}   \
    --window-size {WINDOW_SIZE} \
    --epochs      25           \
    --batch-size  512          \
    --num-workers 2

Device: cuda
Loading training data ...


^C


In [4]:
# ── Cell 5: W&B login (one-time per session) ───────────────────────────────
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aszatrowski (aszatrowski-university-of-chicago) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# ── Cell 6: sweep config ───────────────────────────────────────────────────
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'window_size': {'values': [1000, 2000]},
        'lr':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'conv_arch':   {'values': [
            # undilated — ERF ~23–55 SNPs
            '32,64_7,5',
            '64,128_7,5',
            '32,64,128_7,5,3',
            # dilated — exponential dilation expands ERF to cover the full window
            '32,64_7,7_1,4',           # ERF ~103 SNPs
            '64,128_7,7_1,4',
            '32,64,128_7,7,7_1,4,16',  # ERF ~1600 SNPs
            '64,128,256_7,7,7_1,4,16',
        ]},
        'global_pool': {'values': [True, False]},
        'dropout':     {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
    },
}

sweep_id = wandb.sweep(sweep_config, project='ancestry_cnn')
print('Sweep ID:', sweep_id)

Create sweep with ID: 8b34z4x1
Sweep URL: https://wandb.ai/aszatrowski-university-of-chicago/ancestry_cnn/sweeps/8b34z4x1
Sweep ID: 8b34z4x1


In [6]:
# ── Cell 7: run sweep agent (needs Cells 1, 3, 5, 6) ─────────────────────
import sys
sys.path.insert(0, f'{REPO}/scripts')
# Evict both modules so Python re-reads the updated files from disk
sys.modules.pop('train', None)
sys.modules.pop('model', None)
from train import run_training, parse_conv_arch

SWEEP_EPOCHS = 10   # short runs during search; best config retrained with 25 epochs below
SWEEP_COUNT  = 20   # total trials (~3-4 h on T4)

def sweep_fn():
    with wandb.init() as run:
        w = dict(run.config)
        channels, kernels, dilations = parse_conv_arch(w['conv_arch'])
        cfg = {
            'window_size':    w['window_size'],
            'lr':             w['lr'],
            'conv_channels':  channels,
            'kernel_sizes':   kernels,
            'dilation_rates': dilations,
            'global_pool':    w['global_pool'],
            'dropout':        w['dropout'],
            'epochs':         SWEEP_EPOCHS,
            'batch_size':     512,
            'num_workers':    2,
            'use_wandb':      True,
        }
        run_training(cfg, DATA_H5, None, device)  # output=None skips checkpoint saving

wandb.agent(sweep_id, sweep_fn, count=SWEEP_COUNT)

wandb: Agent Starting Run: gizxae79 with config:
wandb: 	conv_arch: 32,64,128_7,5,3
wandb: 	dropout: 0.4024470561253398
wandb: 	global_pool: False
wandb: 	lr: 0.0008994509270242007
wandb: 	window_size: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 148,920  |  val windows: 31,960
  params: 1,053,061
Epoch   1/10  loss=1.5015  train_acc=0.3115  val_acc=0.3276
Epoch   2/10  loss=1.2917  train_acc=0.3844  val_acc=0.3933
Epoch   3/10  loss=1.2131  train_acc=0.4244  val_acc=0.4928
Epoch   4/10  loss=1.1244  train_acc=0.4825  val_acc=0.4497
Epoch   5/10  loss=1.0576  train_acc=0.5234  val_acc=0.5903
Epoch   6/10  loss=1.0079  train_acc=0.5488  val_acc=0.4604
Epoch   7/10  loss=0.9695  train_acc=0.5714  val_acc=0.6616
Epoch   8/10  loss=0.9471  train_acc=0.5838  val_acc=0.6665
Epoch   9/10  loss=0.9270  train_acc=0.5935  val_acc=0.6638
Epoch  10/10  loss=0.9158  train_acc=0.5983  val_acc=0.6746

Best val accuracy: 0.6746


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▅▅▃▃▂▂▁▁▁
train_acc,▁▃▄▅▆▇▇███
val_acc,▁▂▄▃▆▄████
best_val_acc,0.67456
epoch,10
loss,0.91578
train_acc,0.59833
val_acc,0.67456


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: v9r0zgd2 with config:
wandb: 	conv_arch: 32,64,128_7,5,3
wandb: 	dropout: 0.1137302265213041
wandb: 	global_pool: False
wandb: 	lr: 0.004250302876302715
wandb: 	window_size: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 297,840  |  val windows: 63,920
  params: 528,773
Epoch   1/10  loss=1.5652  train_acc=0.2824  val_acc=0.3605
Epoch   2/10  loss=1.3359  train_acc=0.3619  val_acc=0.4753
Epoch   3/10  loss=1.2484  train_acc=0.4292  val_acc=0.5212
Epoch   4/10  loss=1.1798  train_acc=0.4780  val_acc=0.5649
Epoch   5/10  loss=1.1346  train_acc=0.5032  val_acc=0.5608
Epoch   6/10  loss=1.0964  train_acc=0.5221  val_acc=0.5472
Epoch   7/10  loss=1.0695  train_acc=0.5361  val_acc=0.5995
Epoch   8/10  loss=1.0450  train_acc=0.5494  val_acc=0.5838
Epoch   9/10  loss=1.0264  train_acc=0.5604  val_acc=0.6149
Epoch  10/10  loss=1.0117  train_acc=0.5681  val_acc=0.6258

Best val accuracy: 0.6258


...

  train windows: 148,920  |  val windows: 31,960
  params: 32,773
Epoch   1/10  loss=1.4826  train_acc=0.3130  val_acc=0.2759
Epoch   2/10  loss=1.3964  train_acc=0.3494  val_acc=0.2185
Epoch   3/10  loss=1.3462  train_acc=0.3664  val_acc=0.2803
Epoch   4/10  loss=1.3012  train_acc=0.3838  val_acc=0.2278
Epoch   5/10  loss=1.2741  train_acc=0.3968  val_acc=0.3849
Epoch   6/10  loss=1.2465  train_acc=0.4125  val_acc=0.2729
Epoch   7/10  loss=1.2301  train_acc=0.4234  val_acc=0.4107
Epoch   8/10  loss=1.2097  train_acc=0.4366  val_acc=0.2793
Epoch   9/10  loss=1.1994  train_acc=0.4439  val_acc=0.4339
Epoch  10/10  loss=1.1893  train_acc=0.4487  val_acc=0.4792

Best val accuracy: 0.4792


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▆▅▄▃▂▂▁▁▁
train_acc,▁▃▄▅▅▆▇▇██
val_acc,▃▁▃▁▅▂▆▃▇█
best_val_acc,0.47922
epoch,10
loss,1.18934
train_acc,0.44866
val_acc,0.47922


wandb: Agent Starting Run: ebouxqls with config:
wandb: 	conv_arch: 32,64,128_7,5,3
wandb: 	dropout: 0.299504274336282
wandb: 	global_pool: False
wandb: 	lr: 0.00038210022883973386
wandb: 	window_size: 1000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


  train windows: 297,840  |  val windows: 63,920
  params: 528,773
Epoch   1/10  loss=1.3715  train_acc=0.3813  val_acc=0.5117
Epoch   2/10  loss=1.1356  train_acc=0.5059  val_acc=0.5399
Epoch   3/10  loss=1.0417  train_acc=0.5551  val_acc=0.5780
Epoch   4/10  loss=0.9808  train_acc=0.5845  val_acc=0.6066
Epoch   5/10  loss=0.9410  train_acc=0.6047  val_acc=0.6228
Epoch   6/10  loss=0.9039  train_acc=0.6228  val_acc=0.6216
Epoch   7/10  loss=0.8781  train_acc=0.6354  val_acc=0.6097
Epoch   8/10  loss=0.8587  train_acc=0.6449  val_acc=0.6533
Epoch   9/10  loss=0.8422  train_acc=0.6535  val_acc=0.6589
Epoch  10/10  loss=0.8387  train_acc=0.6544  val_acc=0.6591

Best val accuracy: 0.6591


epoch,▁▂▃▃▄▅▆▆▇█
loss,█▅▄▃▂▂▂▁▁▁
train_acc,▁▄▅▆▇▇████
val_acc,▁▂▄▆▆▆▆███
best_val_acc,0.65911
epoch,10
loss,0.83869
train_acc,0.65437
val_acc,0.65911


In [7]:
# ── Cell 8: retrain best config for full 25 epochs ────────────────────────
api   = wandb.Api()
sweep = api.sweep(f'aszatrowski-university-of-chicago/ancestry_cnn/{sweep_id}')
best  = max(sweep.runs, key=lambda r: r.summary.get('val_acc', 0))
print('Best run:', best.name)
print('Config:  ', dict(best.config))
print(f'val_acc:  {best.summary["val_acc"]:.4f}')

bc = dict(best.config)
channels, kernels, dilations = parse_conv_arch(bc['conv_arch'])
best_cfg = {
    'window_size':    bc['window_size'],
    'lr':             bc['lr'],
    'conv_channels':  channels,
    'kernel_sizes':   kernels,
    'dilation_rates': dilations,
    'global_pool':    bc['global_pool'],
    'dropout':        bc['dropout'],
    'epochs':         25,
    'batch_size':     512,
    'num_workers':    2,
    'use_wandb':      True,
}
run_training(best_cfg, DATA_H5, CKPT_OUT, device)

Best run: lively-sweep-17
Config:   {'lr': 0.00016165813143785072, 'dropout': 0.32259891952928066, 'conv_arch': '32,64_7,5', 'global_pool': False, 'window_size': 2000}
val_acc:  0.6987
  train windows: 148,920  |  val windows: 31,960
  params: 2,060,293
Epoch   1/25  loss=1.4743  train_acc=0.3381  val_acc=0.4895


Error: You must call wandb.init() before wandb.log()

In [ ]:
# ── Cell 9: evaluate (needs Cell 1 only — safe after a restart) ───────────
!python {REPO}/scripts/evaluate.py \
    --data        {DATA_H5}       \
    --admixed     {ADMIXED_H5}    \
    --checkpoint  {CKPT_OUT}      \
    --confusion   {CONFUSION_OUT} \
    --karyogram   {KARYOGRAM_OUT} \
    --window-size {WINDOW_SIZE}

In [ ]:
# ── Cell 10: display results (needs Cell 1 only) ──────────────────────────
from IPython.display import Image, display
display(Image(CONFUSION_OUT))
display(Image(KARYOGRAM_OUT))